# ML-04: Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MahmooudEssam/FlyRank-Intern/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order**, each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**My lane:** help editors decide which page-2 pages to review first.

**1. What one row means.** In my feature frame, one row is one page (client x content) at the decision moment, the start of 2026-03-15. The raw table it is built from has one row per page per day.

**2. Tables I use.** Only `fact_content_daily_performance`, partition `month=2026-03`.

**3. Time window.** Everything stays inside March 2026:

| Window | Dates | Used for |
|---|---|---|
| Features | 2026-03-01 to 2026-03-14 | what the model may see |
| Label | 2026-03-15 to 2026-03-28 | the outcome only |

These are 14-day windows, so I halved the W02 thresholds (100 and 30 impressions become 50 and 15). June stays sealed as the final test month.

In [ ]:
import os

import duckdb
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict


# Look for the token in Colab first, then on this computer. It never gets printed.
def get_token():
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except ImportError:
        from huggingface_hub import get_token as saved_token
        return os.environ.get("HF_TOKEN") or saved_token()


con = duckdb.connect()
con.execute(f"CREATE SECRET hf (TYPE huggingface, TOKEN '{get_token()}')")

MARCH_FILES = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Download March once, so the queries below don't keep hitting Hugging Face
con.execute(f"""
    CREATE TEMP TABLE march AS
    SELECT report_date, client_hash_id, content_hash_id,
           gsc_impressions, gsc_clicks, gsc_sum_position, gsc_data_available
    FROM read_parquet('{MARCH_FILES}')
""")
print("March is loaded")

## 2. Fields: feature / label / context / excluded

**4. What I predict.** I rank page-2 pages by their chance of reaching page 1. The label `reached_page1` is 1 when a page has at least 15 impressions and a weighted position of 10 or better from March 15 to 28. It is a proxy: the data has no edit log, so it shows pages that rose on their own, not pages that rose because someone edited them.

**5. What I deliberately exclude.** `pos_t`, the page's position during the label window. The label is calculated from it, and nobody can know it on March 15.

| Bucket | Fields |
|---|---|
| Feature | `pos_f`, `impr_f`, `ctr_f`, `days_with_impr_f`, `pos_change_f` |
| Label | `reached_page1` |
| Context | `client_hash_id`, `content_hash_id` (for joining and splitting only) |
| Excluded | `pos_t` (future information), all GA4 columns (I don't need them for this first version) |

## 3. Verify it with queries (grain, counts, missing values, windows)

**Query 1: the grain.** If one raw row is really one page per day, no key shows up twice.

In [ ]:
con.sql("""
    WITH keys AS (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
        FROM march
        GROUP BY ALL
    )
    SELECT SUM(n) AS total_rows,
           COUNT(*) AS unique_page_days,
           COUNT(*) FILTER (WHERE n > 1) AS duplicated_page_days
    FROM keys
""").df()

**Query 2: my slice.** Candidates are pages with at least 50 impressions and a weighted position above 10 and up to 20 during March 1 to 14. This counts their daily rows and shows the dates they cover.

In [ ]:
# Page-2 candidates and their features. Only March 1 to 14 is used here.
CANDIDATES = """
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS pos_f,
           SUM(gsc_impressions)::DOUBLE AS impr_f,
           SUM(gsc_clicks)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS ctr_f,
           COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impr_f,
           -- above 0 means the page moved up between week 1 and week 2
           SUM(gsc_sum_position) FILTER (WHERE report_date < DATE '2026-03-08')::DOUBLE
             / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date < DATE '2026-03-08'), 0)
           - SUM(gsc_sum_position) FILTER (WHERE report_date >= DATE '2026-03-08')::DOUBLE
             / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date >= DATE '2026-03-08'), 0) AS pos_change_f
    FROM march
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'
    GROUP BY client_hash_id, content_hash_id
    HAVING impr_f >= 50 AND pos_f > 10 AND pos_f <= 20
"""

con.sql(f"""
    SELECT COUNT(*) AS slice_rows,
           COUNT(DISTINCT (client_hash_id, content_hash_id)) AS pages,
           COUNT(DISTINCT client_hash_id) AS clients,
           MIN(report_date) AS first_date,
           MAX(report_date) AS last_date
    FROM march
    JOIN ({CANDIDATES}) USING (client_hash_id, content_hash_id)
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-28'
""").df()

**Query 3: availability.** How many March rows survive when I keep only `gsc_data_available IS TRUE`?

In [ ]:
con.sql("""
    SELECT COUNT(*) AS all_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_surviving,
           ROUND(100.0 * COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) / COUNT(*), 2) AS pct_surviving
    FROM march
""").df()

### Five features

| Feature | Meaning | Available when? |
|---|---|---|
| `pos_f` | weighted average position | Knowable at the decision moment because it only uses March 1 to 14. |
| `impr_f` | total impressions | Knowable at the decision moment because those impressions were already recorded by March 14. |
| `ctr_f` | clicks divided by impressions | Knowable at the decision moment because both counts stop at March 14. |
| `days_with_impr_f` | days with at least one impression | Knowable at the decision moment because it counts days that have already passed. |
| `pos_change_f` | week 1 position minus week 2 position | Knowable at the decision moment because both weeks end by March 14. |

In [ ]:
frame = con.sql(f"""
    WITH label_window AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impr_t,
               SUM(gsc_sum_position)::DOUBLE / NULLIF(SUM(gsc_impressions), 0) AS pos_t
        FROM march
        WHERE gsc_data_available IS TRUE
          AND report_date BETWEEN DATE '2026-03-15' AND DATE '2026-03-28'
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT c.*, l.pos_t,
           CASE WHEN l.impr_t >= 15 AND l.pos_t <= 10 THEN 1 ELSE 0 END AS reached_page1
    FROM ({CANDIDATES}) c
    LEFT JOIN label_window l USING (client_hash_id, content_hash_id)
""").df()

FEATURES = ["pos_f", "impr_f", "ctr_f", "days_with_impr_f", "pos_change_f"]

# One row must mean one page
assert not frame.duplicated(["client_hash_id", "content_hash_id"]).any()

print(f"{len(frame):,} pages, {frame['reached_page1'].mean():.1%} reached page 1")
frame[FEATURES + ["reached_page1"]].describe().T.round(2)

### The trap: leaking the label on purpose

I score the five features first. Then I add `pos_t`, which the label is built from, and score again. Then I delete it and score one last time. Pages from the same client never sit in both training and testing.

The scores are ROC AUC (1.0 is perfect, 0.5 is guessing) and precision@50 (how many of the top 50 pages really reached page 1).

In [ ]:
def quick_score(columns):
    model = HistGradientBoostingClassifier(random_state=42)
    scores = cross_val_predict(model, frame[columns], frame["reached_page1"],
                               groups=frame["client_hash_id"], cv=GroupKFold(5),
                               method="predict_proba")[:, 1]
    top_50 = frame.loc[pd.Series(scores, index=frame.index).nlargest(50).index]
    return {"roc_auc": roc_auc_score(frame["reached_page1"], scores),
            "precision_at_50": top_50["reached_page1"].mean()}


results = {"1. five honest features": quick_score(FEATURES)}
results["2. with the leak (pos_t)"] = quick_score(FEATURES + ["pos_t"])

# Delete the leak so nothing later can use it by accident
frame = frame.drop(columns="pos_t")
results["3. leak deleted (the number I keep)"] = quick_score(FEATURES)

pd.DataFrame(results).T.round(3)

## 4. Data limits

**My named limitation: a few clients hold much of this slice.** Pages from the same client tend to move together, so the score can reflect those clients more than page-2 pages in general. The numbers below show how concentrated the slice is. This is also one month only, so I treat the score as directional.

In [ ]:
pages_per_client = frame["client_hash_id"].value_counts()
print(f"Clients in the slice: {len(pages_per_client)}")
print(f"Largest client: {pages_per_client.iloc[0] / len(frame):.1%} of pages")
print(f"Top 3 clients: {pages_per_client.iloc[:3].sum() / len(frame):.1%} of pages")

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled: markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.